In [35]:
import numpy as np
from scipy.optimize import differential_evolution
from scipy.special import expit

In [38]:
def rw_transfer_llf(para,cue,outcome,action,q0=.5):
    
    """
    Transfer learning RW model
    
    args:
        para: parameter
        
        cue: happy/angry face, +0.5 denotes happy face, -0.5 denotes angry face
        
        outcome: reward/punishement, 1 denotes reward outcome, 0 denotes punishement outcome
        
        action: appraoch/avoidance, 1 denotes appraoch action, 0 denotes avoid action
        q0 : initial q value
    
    
    """
    
    alpha,tau,bias0,bias1,ep = para
    
    
    trial_len = len(action)
    nll = 0
    q_u = q0
    
    for t in range(trial_len):
        
        cue_t = cue[t]
        a_t = action[t]
        r_t = outcome[t]
        
        q_t = q_u if cue_t>0 else 1-q_u
        
        ## decision: e-softmax
        
        p_app = expit(tau * (2*q_t - 1) + bias0 + bias1 * cue_t)
        p_app = (1-ep) * p_app + ep/2
        
        
        # log-likelihood
        if a_t == 1:
            p_choice = p_app
        else:
            p_choice = 1 - p_app
        p_choice = np.clip(p_choice, 1e-8, 1 - 1e-8)
        nll -= np.log(p_choice)
        
        
        ## align outcome to happy face appraoch action
        
        rev_outcome = r_t if a_t == 1 else 1 - r_t
        cue_outcome = rev_outcome if cue_t > 0 else 1 - rev_outcome
        
        ## update
        
        q_u  += alpha * (cue_outcome - q_u)
    
    return nll,q_u


def rw_seprate_llf(para,cue,outcome,action,q0=.5):
    """
    Separate cue learning RW model
    
    args:
        para: parameter
        
        cue: happy/angry face, +0.5 denotes happy face, -0.5 denotes angry face
        
        outcome: reward/punishement, 1 denotes reward outcome, 0 denotes punishement outcome
        
        action: appraoch/avoidance, 1 denotes appraoch action, 0 denotes avoid action
    
    
    """
    
    alpha,tau,bias0,bias1,ep = para
    
    
    trial_len = len(action)
    nll = 0
    q_h = q0  ## happy face
    q_a = 1-q0  ## angry face
    
    for t in range(trial_len):
        
        cue_t = cue[t]
        a_t = action[t]
        r_t = outcome[t]
        
        q_t = q_h if cue_t>0 else q_a
        
        ## decision: e-softmax
        
        p_app = expit(tau * (2*q_t - 1) + bias0 + bias1 * cue_t)
        p_app = (1-ep) * p_app + ep/2
        
        # log-likelihood
        if a_t == 1:
            p_choice = p_app
        else:
            p_choice = 1 - p_app
        p_choice = np.clip(p_choice, 1e-8, 1 - 1e-8)
        nll -= np.log(p_choice)
        
        
        ## align outcome to happy face appraoch action
        
        rev_outcome = r_t if a_t == 1 else 1 - r_t
        
        ## update
        
        pe = rev_outcome - q_t
        if cue[t] > 0:
            q_h += alpha*pe
        else:
            q_a += alpha*pe
    
    q_u = q_h
    return nll, q_u

def fit_model(data):
    """
    Fit transfer learning and separate cue learning model

    """

    bounds = [
        (1e-4, 0.9999),  # alpha
        (1e-4, 20.0),    # tau
        (-8.0, 8.0),     # bias0
        (-8.0, 8.0),     # bias1
        (1e-4, 0.999)    # ep
    ]

    q0_transfer = 0.5
    q0_separate = 0.5

    results = {}

    block_cols = ['block1', 'block2', 'block3']

    for i, bcol in enumerate(block_cols, start=1):
        mask = data[bcol] == 1
        cue = data.loc[mask, 'cue'].values
        outcome = data.loc[mask, 'outcome'].values
        action = data.loc[mask, 'choice'].values

        t_result = differential_evolution(
            rw_transfer_nll_only,
            bounds=bounds,
            args=(cue, outcome, action, q0_transfer),
            maxiter=1000,
            popsize=30,
            polish=True,
            disp=True
        )
        t_para = t_result.x
        t_nll = t_result.fun

        _, q_end_transfer = rw_transfer_llf(
            t_para, cue, outcome, action, q0=q0_transfer
        )
        # the last block q value as the initial q value of next block
        q0_transfer = q_end_transfer

        s_result = differential_evolution(
            rw_separate_nll_only,
            bounds=bounds,
            args=(cue, outcome, action, q0_separate),
            maxiter=1000,
            popsize=30,
            polish=True,
            disp=True
        )
        s_para = s_result.x
        s_nll = s_result.fun

        _, q_end_separate = rw_seprate_llf(
            s_para, cue, outcome, action, q0=q0_separate
        )
        q0_separate = q_end_separate

        results[f'block{i}_t_para'] = t_para
        results[f'block{i}_t_nll'] = t_nll
        results[f'block{i}_s_para'] = s_para
        results[f'block{i}_s_nll'] = s_nll
        results[f'block{i}_t_q_end'] = q_end_transfer
        results[f'block{i}_s_q_end'] = q_end_separate

    return results
def rw_transfer_nll_only(para, cue, outcome, action, q0):
    nll, _ = rw_transfer_llf(para, cue, outcome, action, q0=q0)
    return nll

def rw_separate_nll_only(para, cue, outcome, action, q0):
    nll, _ = rw_seprate_llf(para, cue, outcome, action, q0=q0)
    return nll


In [14]:
## fit models to excluded and included participants
import pandas as pd
ex_dat = pd.read_csv('excluded_stan_fit.csv')
in_dat = pd.read_csv('included_session1_stan_fit.csv')

In [15]:
ex_dat['block2'] = ex_dat['block2']/2
in_dat['block2'] = in_dat['block2']/2

In [32]:
ex_grouped = (
    ex_dat.groupby('sub_id')[['choice', 'cue', 'outcome','block1','block2','block3']]
    .apply(lambda df: {
        'choice': df['choice'].tolist(),
        'cue': df['cue'].tolist(),
        'outcome': df['outcome'].tolist(),
        'block1':df['block1'].tolist(),
        'block2':df['block2'].tolist(),
        'block3':df['block3'].tolist()        
    })
    .to_dict()
)
in_grouped = (
    in_dat.groupby('sub_id')[['choice','cue','outcome','block1','block2','block3']]
    .apply(lambda df:{
        'choice':df['choice'].tolist(),
        'cue':df['cue'].tolist(),
        'outcome':df['outcome'].tolist(),
        'block1':df['block1'].tolist(),
        'block2':df['block2'].tolist(),
        'block3':df['block3'].tolist()   
    }).to_dict()
)

In [39]:
save_ex_fit = {}
save_in_fit = {}

for i,v in ex_grouped.items():
    save_ex_fit[i] = fit_model(pd.DataFrame(v))
for i,v in in_grouped.items():
    save_in_fit[i] = fit_model(pd.DataFrame(v))

differential_evolution step 1: f(x)= 121.42990626754376
differential_evolution step 2: f(x)= 121.42990626754376
differential_evolution step 3: f(x)= 121.42990626754376
differential_evolution step 4: f(x)= 121.42990626754376
differential_evolution step 5: f(x)= 121.42990626754376
differential_evolution step 6: f(x)= 120.8407314445254
differential_evolution step 7: f(x)= 119.78286979339633
differential_evolution step 8: f(x)= 119.78286979339633
differential_evolution step 9: f(x)= 119.78286979339633
differential_evolution step 10: f(x)= 119.12274957303926
differential_evolution step 11: f(x)= 118.73453289456907
differential_evolution step 12: f(x)= 118.64441627349586
differential_evolution step 13: f(x)= 118.4248920368477
differential_evolution step 14: f(x)= 118.4248920368477
differential_evolution step 15: f(x)= 118.4248920368477
differential_evolution step 16: f(x)= 118.35442655702484
differential_evolution step 17: f(x)= 118.35442655702484
differential_evolution step 18: f(x)= 118.35

In [57]:
pd.DataFrame({"block1 separate cue":pd.DataFrame(save_ex_fit).loc['block1_s_nll',:],
             "block1 transfer":pd.DataFrame(save_ex_fit).loc['block1_t_nll',:]})

,block1 separate cue,block1 transfer
1,118.518104,116.703302
2,133.268425,129.855966
3,134.543192,135.591844
4,119.89584,103.518885
5,28.14637,29.014167
6,139.006804,139.204655
7,126.983002,133.251091
8,103.441093,102.470881
9,133.508618,137.06241


In [58]:
pd.DataFrame({"block2 separate cue":pd.DataFrame(save_ex_fit).loc['block2_s_nll',:],
             "block2 transfer":pd.DataFrame(save_ex_fit).loc['block2_t_nll',:]})

,block2 separate cue,block2 transfer
1,132.833112,137.735991
2,129.624654,114.206665
3,148.731339,148.104786
4,86.102033,85.138973
5,63.654477,64.597919
6,139.674954,142.863869
7,143.481503,146.956574
8,142.220305,138.012034
9,140.758165,141.510734


In [64]:
pd.DataFrame({"block3 separate cue":pd.DataFrame(save_ex_fit).loc['block3_s_nll',:],
             "block3 transfer":pd.DataFrame(save_ex_fit).loc['block3_t_nll',:]})

,block3 separate cue,block3 transfer
1,100.539233,96.874868
2,130.061707,125.857246
3,152.551701,150.979081
4,79.901836,72.968356
5,33.276359,31.355552
6,144.312449,147.109442
7,148.651448,150.251153
8,138.429228,141.292623
9,139.362422,138.797467


In [62]:
pd.DataFrame({"block1 separate cue":pd.DataFrame(save_in_fit).loc['block1_s_nll',:],
             "block1 transfer":pd.DataFrame(save_in_fit).loc['block1_t_nll',:]}).median()

block1 separate cue    110.624149
block1 transfer          98.89945
dtype: object

In [61]:
pd.DataFrame({"block2 separate cue":pd.DataFrame(save_in_fit).loc['block2_s_nll',:],
             "block2 transfer":pd.DataFrame(save_in_fit).loc['block2_t_nll',:]}).median()

block2 separate cue    107.900159
block2 transfer        101.753168
dtype: object

In [60]:
pd.DataFrame({"block3 separate cue":pd.DataFrame(save_in_fit).loc['block3_s_nll',:],
             "block3 transfer":pd.DataFrame(save_in_fit).loc['block3_t_nll',:]}).median()

block3 separate cue    111.138012
block3 transfer         98.103588
dtype: object

In [50]:
pd.DataFrame({"sum separate cue":pd.DataFrame(save_in_fit).loc['block1_s_nll',:]+pd.DataFrame(save_in_fit).loc['block2_s_nll',:]+pd.DataFrame(save_in_fit).loc['block3_s_nll',:],
             "sum transfer":pd.DataFrame(save_in_fit).loc['block1_t_nll',:]+pd.DataFrame(save_in_fit).loc['block2_t_nll',:]+pd.DataFrame(save_in_fit).loc['block3_t_nll',:]})

,sum separate cue,sum transfer
1,337.367564,282.867056
2,330.669922,239.916157
3,321.560524,311.503818
4,321.271507,335.9144
5,348.126724,324.631246
6,355.493225,213.27708
7,342.997479,312.552553
8,383.036402,353.548164
9,270.723882,215.029538
10,206.667642,305.415618


In [85]:
## check the ep parameter
block1_ep = []
block2_ep = []
block3_ep = []
for i,v in save_ex_fit.items():
    for x,y in v.items():
        if 't_para' in x:
            if 'block1' in x:
                block1_ep.append(y[-1])
            elif 'block2' in x:
                block2_ep.append(y[-1])
            elif 'block3' in x:
                block3_ep.append(y[-1])
1-pd.DataFrame({'block1 ep':block1_ep,
             'block2 ep':block2_ep,
             'block3 ep':block3_ep})


,block1 ep,block2 ep,block3 ep
0,0.999900,0.384412,0.822322
1,0.383988,0.665749,0.563676
2,0.382962,0.240539,0.218580
3,0.600235,0.240368,0.999900
4,0.999900,0.918947,0.991370
5,0.999900,0.282189,0.206265
6,0.999900,0.251365,0.256358
7,0.999900,0.339371,0.096345
8,0.441287,0.338007,0.337729


In [86]:
block1_ep = []
block2_ep = []
block3_ep = []
for i,v in save_in_fit.items():
    for x,y in v.items():
        if 't_para' in x:
            if 'block1' in x:
                block1_ep.append(y[-1])
            elif 'block2' in x:
                block2_ep.append(y[-1])
            elif 'block3' in x:
                block3_ep.append(y[-1])
1-pd.DataFrame({'block1 ep':block1_ep,
             'block2 ep':block2_ep,
             'block3 ep':block3_ep})

,block1 ep,block2 ep,block3 ep
0,0.781772,0.879096,0.627933
1,0.888039,0.999900,0.972585
2,0.780266,0.700073,0.663497
3,0.585950,0.748546,0.472054
4,0.999900,0.844086,0.736472
5,0.870620,0.851018,0.726760
6,0.999900,0.999900,0.840242
7,0.938539,0.361943,0.232824
8,0.953460,0.982011,0.914519
9,0.999900,0.999900,0.855734
